In [1]:
# ==============================================================================
# GERADOR SINTÉTICO DE DADOS - SAC MÓVEIS RESIDENCIAIS
# ==============================================================================
import pandas as pd
import random

templates = {
    'vendas': {
        's': ['', 'Olá', 'Bom dia', 'Gostaria de saber', 'Por favor'],
        'a': ['quero comprar', 'qual o preco do', 'tem cupom para', 'como faco para adquirir', 'desejo orcamento de'],
        'o': ['sofa retratil 3 lugares', 'conjunto de mesa de jantar', 'guarda roupa casal', 'painel para tv', 'colchao queen size']
    },
    'suporte': {
        's': ['', 'Oi', 'Preciso de ajuda', 'Por gentileza', 'Socorro'],
        'a': ['como montar o', 'onde baixo o manual do', 'estou com duvida no', 'veio faltando parafuso no', 'preciso de assistencia para'],
        'o': ['armario de cozinha', 'rack da sala', 'berco do bebe', 'esquema de montagem', 'manual da estante']
    },
    'trocas_devolucoes': {
        's': ['', 'Olá', 'Por favor', 'Gostaria de solicitar', 'Quero abrir'],
        'a': ['preciso trocar o', 'quero devolver a', 'como solicito o estorno do', 'desejo solicitar a troca da', 'como funciona a devolucao do'],
        'o': ['produto com defeito', 'mesa que veio arranhada', 'cadeira no prazo de 7 dias', 'pedido cancelado', 'item com avaria']
    },
    'reclamacoes': {
        's': ['', 'Urgente', 'Pessimo atendimento', 'Absurdo', 'Quero registrar'],
        'a': ['estou indignado com o', 'quero fazer uma queixa do', 'estou reclamando do', 'produto veio quebrado e o', 'atendimento horrivel do'],
        'o': ['atraso na minha entrega', 'servico de montagem', 'sac que nao responde', 'pos venda da loja', 'estado do meu movel']
    },
    'logistica_entregas': {
        's': ['', 'Olá', 'Bom dia', 'Por gentileza', 'Preciso saber'],
        'a': ['onde esta o meu', 'qual o prazo de entrega do', 'como rastreio a', 'qual a transportadora do', 'quando chega o'],
        'o': ['meu pedido', 'codigo de rastreamento', 'movel comprado', 'status do envio', 'agendamento da entrega']
    }
}

amostras = []
random.seed(42)

for intencao, comp in templates.items():
    for _ in range(20):  # Total: 100 amostras (20 por classe)
        s = random.choice(comp['s'])
        a = random.choice(comp['a'])
        o = random.choice(comp['o'])
        frase = f"{s} {a} {o}".strip().capitalize()
        amostras.append({'texto': frase, 'intencao': intencao})

df_moveis = pd.DataFrame(amostras)
df_moveis.to_csv('dataset_moveis_100.csv', index=False, encoding='utf-8')

print(" Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!")


 Dataset 'dataset_moveis_100.csv' criado com 100 frases distribuidas em 5 intencoes!


In [3]:
# ==============================================================================
# ATIVIDADE 1: CHATBOT VERSÃO 1 (KNN)
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

# 1. Carregar dataset do CSV
df = pd.read_csv('dataset_moveis_100.csv')

# 2. Divisão Treino e Teste
X_train, X_test, y_train, y_test = train_test_split(
    df['texto'], df['intencao'], test_size=0.30, random_state=42, stratify=df['intencao']
)

# TODO 1: Monte a Pipeline utilizando TfidfVectorizer e KNeighborsClassifier(n_neighbors=3, metric='cosine')
pipeline_knn = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', KNeighborsClassifier(
        n_neighbors=3,
        metric='cosine'
    ))
])

# TODO 2: Treine a pipeline com os dados de treino (X_train, y_train)
pipeline_knn.fit(X_train, y_train)

# TODO 3: Gere as predicoes nos dados de teste e exiba o classification_report e a confusion_matrix
y_pred = pipeline_knn.predict(X_test)

print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))

print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))

LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES  ===")

for i in range(1, 11):
    print(f"\n[Teste {i}/10]")

    # TODO 4: Solicite a frase do usuario via teclado
    frase = input("Digite aqui sua dúvida: ").strip()

    # TODO 5: Extraia as probabilidades e a classe prevista usando predict_proba e predict
    probs = pipeline_knn.predict_proba([frase])

    maior_prob = np.max(probs)

    intencao = pipeline_knn.predict([frase])[0]

    # TODO 6: Aplique a regra de decisao:
    if maior_prob >= LIMIAR_CONFIANCA:

        print(f"Intenção identificada: {intencao}")
        print(f"Probabilidade: {maior_prob:.2%}")

    else:

        print("Fallback: não foi possível identificar a intenção com segurança.")
        print("Encaminhando para atendimento humano.")

    pass


=== RELATÓRIO DE CLASSIFICAÇÃO ===
                    precision    recall  f1-score   support

logistica_entregas       1.00      1.00      1.00         6
       reclamacoes       1.00      1.00      1.00         6
           suporte       1.00      1.00      1.00         6
 trocas_devolucoes       1.00      1.00      1.00         6
            vendas       1.00      1.00      1.00         6

          accuracy                           1.00        30
         macro avg       1.00      1.00      1.00        30
      weighted avg       1.00      1.00      1.00        30


=== MATRIZ DE CONFUSÃO ===
[[6 0 0 0 0]
 [0 6 0 0 0]
 [0 0 6 0 0]
 [0 0 0 6 0]
 [0 0 0 0 6]]

=== INICIANDO BATERIA DE TESTES  ===

[Teste 1/10]
Digite aqui sua dúvida: tem cupom para compra?
Intenção identificada: vendas
Probabilidade: 100.00%

[Teste 2/10]
Digite aqui sua dúvida: quero um raque
Intenção identificada: reclamacoes
Probabilidade: 66.67%

[Teste 3/10]
Digite aqui sua dúvida: vou tomar refrigerante
Fall

In [ ]:
# ==============================================================================
# ATIVIDADE 2: CHATBOT VERSÃO 2 - DECISION TREE
# ==============================================================================

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


df = pd.read_csv('dataset_moveis_100.csv')


X_train, X_test, y_train, y_test = train_test_split(
    df['texto'],
    df['intencao'],
    test_size=0.30,
    random_state=42,
    stratify=df['intencao']
)



pipeline_tree = Pipeline([
    ('vectorizer', TfidfVectorizer()),
    ('classifier', DecisionTreeClassifier(random_state=42))
])



pipeline_tree.fit(X_train, y_train)




y_pred = pipeline_tree.predict(X_test)



print("\n=== MATRIZ DE CONFUSÃO ===")
print(confusion_matrix(y_test, y_pred))




print("\n=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(classification_report(y_test, y_pred))



# ==============================================================================
# 8. TESTES INTERATIVOS
# ==============================================================================

LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES ===")
print("Digite 8 frases para testar o chatbot.")

for i in range(1, 9):

    print(f"\n[Teste {i}/8]")

    frase = input("Digite a frase do cliente: ").strip()

    # Obter o vetor TF-IDF da frase
    vetor = pipeline_tree.named_steps['vectorizer'].transform([frase])

    # Verificar se existem palavras conhecidas pelo modelo
    palavras_conhecidas = vetor.nnz > 0

    if not palavras_conhecidas:

        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )

    else:

        # Fazer previsão
        intencao = pipeline_tree.predict([frase])[0]

        # Obter probabilidades
        probs = pipeline_tree.predict_proba([frase])

        # Maior probabilidade
        maior_prob = np.max(probs)

        # Aplicar limiar de confiança
        if maior_prob >= LIMIAR_CONFIANCA:

            print(f"Intenção identificada: {intencao}")
            print(f"Probabilidade: {maior_prob:.2%}")

        else:

            print(
                "Desculpe, não entendi sua solicitação. "
                "Encaminhando você para um atendente humano..."
            )# ==============================================================================
# 8. TESTES INTERATIVOS
# ==============================================================================

LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES ===")
print("Digite 8 frases para testar o chatbot.")

for i in range(1, 9):

    print(f"\n[Teste {i}/8]")

    frase = input("Digite a frase do cliente: ").strip()

    # Obter o vetor TF-IDF da frase
    vetor = pipeline_tree.named_steps['vectorizer'].transform([frase])

    # Verificar se existem palavras conhecidas pelo modelo
    palavras_conhecidas = vetor.nnz > 0

    if not palavras_conhecidas:

        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )

    else:

        # Fazer previsão
        intencao = pipeline_tree.predict([frase])[0]

        # Obter probabilidades
        probs = pipeline_tree.predict_proba([frase])

        # Maior probabilidade
        maior_prob = np.max(probs)

        # Aplicar limiar de confiança
        if maior_prob >= LIMIAR_CONFIANCA:

            print(f"Intenção identificada: {intencao}")
            print(f"Probabilidade: {maior_prob:.2%}")

        else:

            print(
                "Desculpe, não entendi sua solicitação. "
                "Encaminhando você para um atendente humano..."
            )# ==============================================================================
# 8. TESTES INTERATIVOS
# ==============================================================================

LIMIAR_CONFIANCA = 0.50

print("\n=== INICIANDO BATERIA DE TESTES ===")
print("Digite 8 frases para testar o chatbot.")

for i in range(1, 9):

    print(f"\n[Teste {i}/8]")

    frase = input("Digite a frase do cliente: ").strip()

    # Obter o vetor TF-IDF da frase
    vetor = pipeline_tree.named_steps['vectorizer'].transform([frase])

    # Verificar se existem palavras conhecidas pelo modelo
    palavras_conhecidas = vetor.nnz > 0

    if not palavras_conhecidas:

        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )

    else:

        # Fazer previsão
        intencao = pipeline_tree.predict([frase])[0]

        # Obter probabilidades
        probs = pipeline_tree.predict_proba([frase])

        # Maior probabilidade
        maior_prob = np.max(probs)

        # Aplicar limiar de confiança
        if maior_prob >= LIMIAR_CONFIANCA:

            print(f"Intenção identificada: {intencao}")
            print(f"Probabilidade: {maior_prob:.2%}")

        else:

            print(
                "Desculpe, não entendi sua solicitação. "
                "Encaminhando você para um atendente humano..."
            )

        print(
            "Desculpe, não entendi sua solicitação. "
            "Encaminhando você para um atendente humano..."
        )


=== MATRIZ DE CONFUSÃO ===
[[4 0 0 0 2]
 [1 4 1 0 0]
 [0 0 6 0 0]
 [0 0 1 5 0]
 [0 0 0 1 5]]

=== RELATÓRIO DE CLASSIFICAÇÃO ===
                    precision    recall  f1-score   support

logistica_entregas       0.80      0.67      0.73         6
       reclamacoes       1.00      0.67      0.80         6
           suporte       0.75      1.00      0.86         6
 trocas_devolucoes       0.83      0.83      0.83         6
            vendas       0.71      0.83      0.77         6

          accuracy                           0.80        30
         macro avg       0.82      0.80      0.80        30
      weighted avg       0.82      0.80      0.80        30


=== INICIANDO BATERIA DE TESTES ===
Digite 8 frases para testar o chatbot.

[Teste 1/8]
Digite a frase do cliente: gustavo inventa desculpa
Desculpe, não entendi sua solicitação. Encaminhando você para um atendente humano...

[Teste 2/8]
Digite a frase do cliente: quero uma beliche
Intenção identificada: trocas_devolucoes
Pr